Chatgpt Cassandra

El script con Faker y Cassandra se trata de una herramienta para generar e insertar datos sintéticos de logs en una base de datos Cassandra, con el objetivo de:

🎯 Propósito principal
Simular datos reales de funcionamiento de un sistema distribuido (como logs de servidores, aplicaciones o microservicios) para:

Pruebas de carga y rendimiento de Cassandra sin usar datos sensibles o reales.

Entrenamiento o prueba de modelos de IA generativa o analítica.

Evaluación de consultas, índices y escalabilidad en entornos distribuidos.

🧱 Componentes del Script
Componente	Descripción
Faker	Biblioteca Python que genera datos falsos pero realistas (IP, texto, fechas, etc.). Ideal para simular tráfico o interacciones humanas.
uuid	Se usa para crear identificadores únicos por log.
random	Asigna aleatoriamente niveles de log como INFO, DEBUG, ERROR.
cassandra-driver	Cliente oficial de Python para conectarse, crear estructuras y hacer consultas a Cassandra.

📋 ¿Qué datos genera?
Cada registro o log contiene:

Campo	Ejemplo	Descripción
id	uuid.uuid4()	Identificador único
timestamp	2025-06-02 14:32:45	Fecha y hora del log
ip	192.168.1.10	IP simulada
level	INFO / DEBUG / ERROR	Nivel de severidad del log
message	"User requested resource"	Mensaje textual aleatorio

🧪 ¿Por qué es útil este tipo de scripts?
Pruebas realistas sin riesgo: No se expone información real del sistema o usuarios.

Carga sintética distribuida: Puedes simular miles de eventos distribuidos por nodos o zonas.

Evaluación de rendimiento: Prueba cómo responde Cassandra ante consultas masivas o escritura intensiva.

In [ ]:
conda install -c conda-forge faker
pip install cassandra-driver

In [ ]:
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement
from faker import Faker
import uuid
import random

# Inicializa el generador de datos falsos
fake = Faker()

# Configura la conexión al clúster de Cassandra
# Reemplaza '127.0.0.1' con la IP de tu nodo Cassandra si es remoto
cluster = Cluster(['127.0.0.1'])
session = cluster.connect()

# Crea el keyspace si no existe
KEYSPACE = "logs"
session.execute(f"""
    CREATE KEYSPACE IF NOT EXISTS {KEYSPACE}
    WITH replication = {{ 'class': 'SimpleStrategy', 'replication_factor': '1' }}
""")

# Usa el keyspace
session.set_keyspace(KEYSPACE)

# Crea la tabla para almacenar logs sintéticos
session.execute("""
    CREATE TABLE IF NOT EXISTS logs_data (
        id UUID PRIMARY KEY,
        timestamp timestamp,
        ip text,
        level text,
        message text
    )
""")

# Niveles de log típicos
log_levels = ['INFO', 'DEBUG', 'ERROR', 'WARNING']

# Función para insertar logs sintéticos
def insert_logs(n=1000):
    for _ in range(n):
        log_id = uuid.uuid4()
        timestamp = fake.date_time_this_year()
        ip = fake.ipv4()
        level = random.choice(log_levels)
        message = fake.sentence()

        query = SimpleStatement("""
            INSERT INTO logs_data (id, timestamp, ip, level, message)
            VALUES (%s, %s, %s, %s, %s)
        """)
        session.execute(query, (log_id, timestamp, ip, level, message))

    print(f"{n} registros de log insertados correctamente en Cassandra.")

# Ejecutar la función
if __name__ == "__main__":
    insert_logs(1000)
